# 🎙️ מערכת תמלול שידור חי בזמן אמת

מערכת מתקדמת לתמלול אוטומטי של שידורים חיים בעברית

## הוראות:
1. הרץ את התא הראשון להתקנה
2. הזן את ה-API Key שלך בתא השני
3. הרץ את התא השלישי להפעלת הממשק
4. לחץ על הקישור שיופיע (*.gradio.live)

---

In [ ]:
# התקנת כל הדרוש
print("📦 מתקין תלויות...")
!pip install -q openai gradio python-dotenv
!apt-get install -y ffmpeg > /dev/null 2>&1

print("\n✅ התקנה הושלמה!\n")
print("בדיקת גרסאות:")
!python --version
!ffmpeg -version | head -1

In [ ]:
# הגדרות - שים את ה-API KEY שלך כאן!
import os

# 👇 החלף את המפתח הזה במפתח שלך מ-https://platform.openai.com/api-keys
OPENAI_API_KEY = "sk-your-openai-api-key-here"

# הגדרת משתנה סביבה
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

if OPENAI_API_KEY == "sk-your-openai-api-key-here":
    print("⚠️  אנא החלף את ה-API Key בתא זה!")
else:
    print("✅ API Key הוגדר")

In [ ]:
# טעינת הקוד מ-GitHub
!git clone https://github.com/<YOUR-USERNAME>/live_transciber.git /content/live_transciber 2>/dev/null || echo "Repository already cloned"
%cd /content/live_transciber

# או להעתיק את הקבצים ישירות
# (כרגע נשתמש בקוד מוטמע למטה)

In [ ]:
# קוד מלא של המערכת
import subprocess
import tempfile
import os
import threading
import time
import logging
from typing import Optional, Dict, List
from datetime import datetime

import openai
from openai import OpenAI
import gradio as gr

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuration
KAN11_URL = "https://r.il.cdn-redge.media/livehls/oil/kancdn-live/live/tmp/kan11/live.livx/playlist.m3u8?renditions"
CHUNK_DURATION = 10
WHISPER_MODEL = "whisper-1"

print("✅ ספריות נטענו")
print(f"OpenAI API Key: {'✅ מוגדר' if os.getenv('OPENAI_API_KEY') else '❌ חסר'}")

In [ ]:
# העתק את הקוד המלא של transcription_engine.py כאן
# (מקוצר לצורך הדוגמה - בגרסה המלאה נעתיק את כל הקוד)

class LiveTranscriptionEngine:
    """מנוע תמלול עם טיפול בשגיאות"""
    
    def __init__(self):
        self.is_running = False
        self.transcription_text = ""
        self.sentence_buffer = []
        self.stats = {'chunks': 0, 'runtime': 0, 'cost': 0, 'errors': 0}
        self.client = None
    
    def start_transcription(self, url: str, chunk_duration: int = 10) -> str:
        if self.is_running:
            return "⚠️ תמלול כבר רץ!"
        
        api_key = os.getenv('OPENAI_API_KEY')
        if not api_key or not api_key.startswith('sk-'):
            return "❌ נא להגדיר OPENAI_API_KEY"
        
        self.client = OpenAI(api_key=api_key)
        self.is_running = True
        self.transcription_text = ""
        self.sentence_buffer = []
        self.stats = {'chunks': 0, 'runtime': 0, 'cost': 0, 'errors': 0}
        self.start_time = time.time()
        
        thread = threading.Thread(
            target=self._worker,
            args=(url, chunk_duration),
            daemon=True
        )
        thread.start()
        
        return "✅ תמלול התחיל!"
    
    def stop_transcription(self) -> str:
        self.is_running = False
        if hasattr(self, 'stream_process') and self.stream_process:
            try:
                self.stream_process.terminate()
            except:
                pass
        return "⏹️ תמלול הופסק"
    
    def _worker(self, url: str, chunk_duration: int):
        try:
            # Start ffmpeg stream
            cmd = [
                'ffmpeg', '-i', url,
                '-f', 'mp3',
                '-acodec', 'libmp3lame',
                '-ar', '16000',
                '-ac', '1',
                '-b:a', '32k',
                '-loglevel', 'error',
                'pipe:1'
            ]
            
            self.stream_process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                bufsize=10**8
            )
            
            time.sleep(3)
            chunk_num = 0
            
            while self.is_running:
                chunk_num += 1
                temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
                temp_filename = temp_file.name
                temp_file.close()
                
                try:
                    # Capture audio
                    approx_size = 16000 * chunk_duration * 2 * 0.125
                    audio_data = self.stream_process.stdout.read(int(approx_size * 1.5))
                    
                    if not audio_data:
                        continue
                    
                    with open(temp_filename, 'wb') as f:
                        f.write(audio_data)
                    
                    if os.path.getsize(temp_filename) < 1000:
                        continue
                    
                    # Transcribe
                    with open(temp_filename, 'rb') as audio_file:
                        transcript = self.client.audio.transcriptions.create(
                            model=WHISPER_MODEL,
                            file=audio_file,
                            language="he"
                        )
                    
                    text = transcript.text.strip()
                    if text:
                        self.sentence_buffer.append(text)
                        if len(self.sentence_buffer) >= 3:
                            self.transcription_text += " ".join(self.sentence_buffer) + "\n\n"
                            self.sentence_buffer = []
                    
                    # Update stats
                    runtime = time.time() - self.start_time
                    self.stats = {
                        'chunks': chunk_num,
                        'runtime': runtime,
                        'cost': (chunk_duration * chunk_num / 60) * 0.006,
                        'errors': self.stats.get('errors', 0)
                    }
                
                except Exception as e:
                    logger.error(f"Error: {e}")
                    self.stats['errors'] = self.stats.get('errors', 0) + 1
                
                finally:
                    if os.path.exists(temp_filename):
                        os.unlink(temp_filename)
        
        except Exception as e:
            logger.error(f"Worker error: {e}")
            self.is_running = False
    
    def get_transcription_text(self) -> str:
        result = self.transcription_text
        if self.sentence_buffer:
            result += " ".join(self.sentence_buffer)
        return result if result else ""
    
    def get_stats(self) -> Dict:
        return self.stats.copy()
    
    def is_active(self) -> bool:
        return self.is_running

# Create engine instance
engine = LiveTranscriptionEngine()
print("✅ מנוע התמלול מוכן!")

In [ ]:
# Gradio UI

def start_ui(use_kan11, custom_url, chunk_duration):
    url = KAN11_URL if use_kan11 else custom_url
    if not url:
        return "❌ נא לבחור כאן 11 או להזין URL", True
    return engine.start_transcription(url, chunk_duration), True

def stop_ui():
    return engine.stop_transcription(), True

def update_transcription():
    text = engine.get_transcription_text()
    if not text:
        return "🎙️ מקליט ומתמלל..." if engine.is_active() else "ממתין לתחילת תמלול..."
    return text

def update_stats():
    stats = engine.get_stats()
    status = "🟢 פעיל" if engine.is_active() else "🔴 מופסק"
    return (
        status,
        f"📦 {stats['chunks']} צ'אנקים",
        f"⏱️ {stats['runtime']:.0f}s",
        f"💰 ${stats['cost']:.4f}",
        f"⚠️ {stats['errors']} שגיאות"
    )

# Build interface
with gr.Blocks(theme=gr.themes.Soft(), title="תמלול לייב") as demo:
    gr.HTML("<h1 style='text-align: center; font-size: 2.5em;'>🎙️ תמלול שידור חי</h1>")
    
    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### 📺 בחירת מקור")
            
            with gr.Row():
                use_kan11 = gr.Checkbox(label="📺 כאן 11", value=True, scale=1)
                custom_url = gr.Textbox(placeholder="או URL אחר...", show_label=False, scale=3)
            
            chunk_duration = gr.Slider(
                minimum=5, maximum=30, value=10, step=1,
                label="משך צ'אנק (שניות)"
            )
            
            with gr.Row():
                start_btn = gr.Button("▶️ התחל", variant="primary", scale=3)
                stop_btn = gr.Button("⏹️ עצור", variant="stop", scale=1)
            
            status_msg = gr.Textbox(show_label=False, interactive=False, visible=False)
            
            gr.Markdown("### 📝 תמלול")
            transcription = gr.Textbox(
                lines=20,
                show_label=False,
                interactive=False,
                show_copy_button=True
            )
        
        with gr.Column(scale=1):
            gr.Markdown("### 📊 סטטוס")
            status = gr.Markdown("🔴 מופסק")
            chunks = gr.Markdown("📦 0")
            runtime = gr.Markdown("⏱️ 0s")
            cost = gr.Markdown("💰 $0.00")
            errors = gr.Markdown("⚠️ 0")
            
            gr.Markdown("""
            ### 💡 הוראות
            1. סמן כאן 11
            2. לחץ התחל
            3. המתן לתמלול
            """)
    
    # Events
    start_btn.click(
        fn=start_ui,
        inputs=[use_kan11, custom_url, chunk_duration],
        outputs=[status_msg, status_msg]
    )
    
    stop_btn.click(
        fn=stop_ui,
        outputs=[status_msg, status_msg]
    )
    
    # Auto-update
    timer1 = gr.Timer(1)
    timer1.tick(fn=update_transcription, outputs=transcription)
    
    timer2 = gr.Timer(5)
    timer2.tick(fn=update_stats, outputs=[status, chunks, runtime, cost, errors])

print("✅ ממשק מוכן!")

In [ ]:
# הפעלת הממשק
print("🚀 מפעיל את הממשק...")
print("הקישור יופיע בעוד כמה שניות...\n")

demo.queue()
demo.launch(
    share=True,  # יצירת URL ציבורי
    debug=False,
    show_error=True
)